# Cleaning: London road speed limits (Ordnance Survey)

> **Licence:** educational use only. OS data must not be pushed to GitHub: `data/raw/digimap/` and `data/clean/digimap/` are in `.gitignore`. **Clear all outputs before committing.**
> Contains OS data © Crown copyright and database rights 2026 Ordnance Survey (100025252).

## 1. Load

In [ ]:
import os

import fiona
import geopandas as gpd
import pandas as pd

RAW_PATH = "../data/raw/digimap/ngd-trn-rami_6570957.gpkg"
CLEAN_PATH = "../data/clean/digimap/road_speed_clean.gpkg"
LOG_PATH = "../logs/cleaning_log_speed_limits.md"
LAYER = "Average And Indicative Speed"

print(fiona.listlayers(RAW_PATH))

In [ ]:
speed = gpd.read_file(RAW_PATH, layer=LAYER)

print(speed.shape)
print(speed.crs)  # expect EPSG:27700 (British National Grid, metres)
print(speed.geometry.geom_type.value_counts())

## 2. Select columns and set up the cleaning log

In [ ]:
KEEP = [
    "osid",                                      # unique road link ID
    "name1_text",                                # street name
    "routehierarchy",                            # road type
    "roadclassificationnumber",                  # e.g. A501
    "indicativespeedlimit_mph",
    "indicativespeedlimit_source",
    "averagespeed_mf7to9indirection_kph",        # Mon–Fri 7–9am (school drop-off)
    "averagespeed_mf7to9againstdirection_kph",
    "averagespeed_mf14to16indirection_kph",      # Mon–Fri 2–4pm (school pick-up)
    "averagespeed_mf14to16againstdirection_kph",
    "geometry_length_m",
    "geometry",
]
speed = speed[KEEP].copy()

# Each cleaning step adds one row to the log table
log = []

def add_log(action, reason, rows):
    log.append({"action": action, "reason": reason, "rows": rows})

In [ ]:
import osmnx as ox

SCHOOL_RADIUS_M = 400

london = ox.geocode_to_gdf("Greater London, UK").to_crs(27700)
london_plus_margin = london.geometry.iloc[0].buffer(SCHOOL_RADIUS_M).simplify(50)

in_area = speed.geometry.intersects(london_plus_margin)
n_outside = (~in_area).sum()
print(f"Road links outside London + {SCHOOL_RADIUS_M}m: {n_outside:,} of {len(speed):,}")

speed = speed[in_area].copy()

add_log(f"Kept only road links within Greater London plus {SCHOOL_RADIUS_M}m",
        f"Download area included parts of Essex, Kent and Surrey; the {SCHOOL_RADIUS_M}m margin keeps roads near border schools",
        int(n_outside))

gpd.GeoDataFrame(geometry=[london_plus_margin], crs=27700).to_file(
    "../data/clean/london_plus_400m.geojson", driver="GeoJSON")

## 3. Look

In [ ]:
print(speed["indicativespeedlimit_mph"].value_counts(dropna=False).sort_index())
print(speed["indicativespeedlimit_source"].value_counts(dropna=False))
print(speed["routehierarchy"].value_counts(dropna=False))
speed.describe()

## 4. Blanks

In [ ]:
n_blank = speed["indicativespeedlimit_mph"].isna().sum()
print(f"No speed limit: {n_blank:,} of {len(speed):,}")

add_log("Checked `indicativespeedlimit_mph` for blanks",
        "A road with no limit can't be counted",
        int(n_blank))

## 5. Repeated road links

In [ ]:
n_dup = speed.duplicated(subset="osid").sum()
print(f"Repeated road links: {n_dup:,}")

speed = speed.drop_duplicates(subset="osid")

add_log("Removed repeated road links (same `osid`)",
        "Each road should be counted once",
        int(n_dup))

## 6. Impossible speed limits

Public roads: 20–70mph ([roads.org.uk](https://www.roads.org.uk/articles/limit/what-speed-limit)). Private roads, car parks and estates can post 5, 10 or 15mph, so those are kept.

In [ ]:
LEGAL_MPH = {20, 30, 40, 50, 60, 70}
PRIVATE_LOW_MPH = {5, 10, 15}

speed["indicativespeedlimit_mph"] = pd.to_numeric(speed["indicativespeedlimit_mph"], errors="coerce")

impossible = (
    speed["indicativespeedlimit_mph"].notna()
    & ~speed["indicativespeedlimit_mph"].isin(LEGAL_MPH | PRIVATE_LOW_MPH)
)
print("Impossible limits:", speed.loc[impossible, "indicativespeedlimit_mph"].value_counts().to_dict())

speed.loc[impossible, "indicativespeedlimit_mph"] = None

add_log("Set speed limits that aren't a UK limit to blank (kept 20–70mph, plus 5/10/15mph private-road limits)",
        "Remove only values that can't be true",
        int(impossible.sum()))

## 7. School-run average speeds

Average the two travel directions (one is often blank on one-way roads), convert kph to mph, and blank values that can't be real.

In [ ]:
KPH_PER_MPH = 1.609
MAX_PLAUSIBLE_MPH = 90

SCHOOL_RUN_SLOTS = {"mf7to9": "Mon–Fri 7–9am", "mf14to16": "Mon–Fri 2–4pm"}

for slot, label in SCHOOL_RUN_SLOTS.items():
    cols = [f"averagespeed_{slot}indirection_kph", f"averagespeed_{slot}againstdirection_kph"]
    out = f"avgspeed_{slot}_mph"

    speed[cols] = speed[cols].apply(pd.to_numeric, errors="coerce")
    speed[out] = (speed[cols].mean(axis=1) / KPH_PER_MPH).round(1)  # mean skips blanks

    add_log(f"Averaged both directions of {label} average speed and converted kph to mph",
            "One comparable speed per road, in the same units as the limits",
            int(speed[out].notna().sum()))

    impossible_avg = (speed[out] <= 0) | (speed[out] > MAX_PLAUSIBLE_MPH)
    speed.loc[impossible_avg, out] = None

    add_log(f"Set {label} average speeds of 0mph or below, or over {MAX_PLAUSIBLE_MPH}mph, to blank",
            "Impossible for a London road",
            int(impossible_avg.sum()))

    add_log(f"Kept roads with no {label} average speed, leaving it blank",
            "Missing traffic data doesn't mean the road isn't there",
            int(speed[out].isna().sum()))

speed[["avgspeed_mf7to9_mph", "avgspeed_mf14to16_mph"]].describe()

## 8. Save the clean data (local only)

In [ ]:
clean = speed.rename(columns={
    "name1_text": "road_name",
    "routehierarchy": "road_type",
    "roadclassificationnumber": "road_number",
    "indicativespeedlimit_mph": "speed_limit_mph",
    "geometry_length_m": "length_m",
})[[
    "osid", "road_name", "road_type", "road_number", "speed_limit_mph",
    "avgspeed_mf7to9_mph", "avgspeed_mf14to16_mph", "length_m", "geometry",
]]

os.makedirs(os.path.dirname(CLEAN_PATH), exist_ok=True)
clean.to_file(CLEAN_PATH, driver="GPKG")
print(f"Saved {len(clean):,} road links to {CLEAN_PATH}")

## 9. Write the cleaning log

In [ ]:
TITLE = "Data Cleaning Log — London Road Speed Limits"

WHY = """We need the speed limit on every road around London primary schools, to test whether schools surrounded by more 20mph roads have fewer child pedestrian collisions.

- **Reliable:** Ordnance Survey National Geographic Database (OS NGD), May 2026, *Average And Indicative Speed* layer. Speed limits are supplied by Basemap, an OS partner (549,422 road links from Basemap, 2,939 from Ordnance Survey and Basemap combined). Average speeds come from in-vehicle GPS (telematics) data.
- **Suitable:** one row per road link, with its speed limit, its road line, its length, its road type and average traffic speeds at school-run times (Mon–Fri 7–9am and 2–4pm).
- **Columns used:** 12 of the layer's 71 columns (ID, name, road type, speed limit and source, school-run average speeds, length, geometry).
- **Area:** Greater London plus a 400m margin, so schools near the border keep all their nearby roads. The London boundary comes from OpenStreetMap (© OpenStreetMap contributors).
- **Feasible:** free for UCL students through Digimap; one 640 MB GeoPackage covering Greater London.
- **Gaps:** speed limits are *indicative* (the limit that applies over most of each road link) and reflect May 2026, so roads that changed to 20mph during 2021–2025 show their current limit. Average speeds are historic averages, and on quiet roads they may be based on few vehicles.
- **Licence:** educational use only; the data itself is not on GitHub. Contains OS data © Crown copyright and database rights 2026 Ordnance Survey (100025252)."""


def write_log(path, title, why, rows):
    lines = [
        f"# {title}", "",
        "## Why we chose this dataset", "", why, "",
        "## Cleaning steps", "",
        "|  | Action | Reason | Rows affected |",
        "|---|---|---|---|",
    ]
    for i, r in enumerate(rows, start=1):
        n = f"{r['rows']:,}" if isinstance(r["rows"], int) else r["rows"]
        lines.append(f"| {i} | {r['action']} | {r['reason']} | {n} |")

    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write("\n".join(lines) + "\n")


write_log(LOG_PATH, TITLE, WHY, log)
print(open(LOG_PATH).read())